# Homework 3 - Machine Learning for Classification

Dataset: 2026 lead-scoring (`course_lead_scoring_2026.csv`). Target: `converted`.

In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

C:\Users\ACER\anaconda3\lib\site-packages\scipy\__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Dataset

Download data (pakai file lokal dari `cohorts/2026/data/` jika ada, kalau tidak download dari GitHub).

In [2]:
local = '../cohorts/2026/data/course_lead_scoring_2026.csv'
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

data = local if os.path.exists(local) else url
df = pd.read_csv(data)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
df.dtypes

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

## Data preparation

* Cek missing values
* Kategorikal -> `'NA'`, numerikal -> `0.0`

In [4]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [5]:
categorical = list(df.dtypes[df.dtypes == 'object'].index)
numerical = [c for c in df.columns if c not in categorical and c != 'converted']

categorical, numerical

(['lead_source', 'industry', 'employment_status', 'location'],
 ['annual_income',
  'number_of_courses_viewed',
  'interaction_count',
  'lead_score'])

In [6]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1

Mode untuk kolom `industry`?

In [7]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [8]:
df.industry.mode()[0]

'technology'

## Question 2

Correlation matrix untuk fitur numerikal.

In [9]:
corr = df[numerical].corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [10]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]

for a, b in pairs:
    print(f'{a} - {b}: {corr.loc[a, b]:.4f}')

interaction_count - lead_score: 0.9157
number_of_courses_viewed - lead_score: 0.7572
number_of_courses_viewed - interaction_count: 0.7216
annual_income - interaction_count: 0.1228


## Split the data

In [11]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [12]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

## Question 3

Mutual information antara `converted` dan variabel kategorikal (training set saja).

In [13]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)

mi = df_train[categorical].apply(mutual_info_converted_score)
mi.round(2).sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

## Question 4

Logistic regression dengan one-hot encoding (`DictVectorizer`).

In [14]:
features = categorical + numerical

def train_and_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[features].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    val_dict = df_val[features].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    y_pred = model.predict(X_val)
    return accuracy_score(y_val, y_pred)

In [15]:
accuracy = train_and_eval(features)
print(accuracy, round(accuracy, 2))

0.645 0.65


## Question 5

Feature elimination: hapus satu fitur, hitung selisih akurasi.

In [16]:
results = []

for f in features:
    subset = [x for x in features if x != f]
    acc = train_and_eval(subset)
    results.append((f, acc, accuracy - acc))

df_elim = pd.DataFrame(results, columns=['feature', 'accuracy', 'difference'])
df_elim.sort_values('difference')

,feature,accuracy,difference
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
2,employment_status,0.645,0.000
3,location,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


In [17]:
candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
df_elim[df_elim.feature.isin(candidates)].sort_values('difference')

,feature,accuracy,difference
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


## Question 6

Regularized logistic regression dengan beberapa nilai `C`.

In [18]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    acc = train_and_eval(features, C=C)
    print(C, round(acc, 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


## Summary

| Question | Answer |
|---|---|
| Q1 | `technology` |
| Q2 | `interaction_count` and `lead_score` (0.916) |
| Q3 | `lead_source` (0.03) |
| Q4 | 0.65 (0.645) |
| Q5 | `number_of_courses_viewed` (diff 0.002) |
| Q6 | 0.001 (0.645) |
